## Time Series Forecast Model

A time series model is a model for data where order in time matters. A forecasting model is a type of time series model whose goal is to predict future values. 

A time series often has these patterns:
- trend
- seasonality
- noise
- autocorrelation

Common forecasting setups:
- One-step forecasting
- Multi-step forecasting
- Univariate forecasting
- Multivariate forecasting

Classical models:
- AR, MA, ARIMA, SARIMA, Exponential Smoothing

Deep Learning models:
- MLP on sliding window, RNN, LSTM, GRU, 1D CNN, Transformer 

In [2]:
import math
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [4]:
torch.manual_seed(42)

T = 1000
t = torch.arange(0, T, dtype=torch.float32)

series = torch.sin(0.02 * t) + 0.3 * torch.sin(0.05 * t) + 0.1 * torch.randn(T)


In [5]:
train_size = int(0.8*T)
train_series = series[:train_size]
test_series = series[train_size:]

In [13]:
class TimeSeriesDataset(Dataset):
    def __init__(self, series, input_window, forecast_horizon=1):
        self.series = series
        self.input_window = input_window
        self.forecast_horizon = forecast_horizon

    def __len__(self):
        return len(self.series) - self.input_window - self.forecast_horizon + 1

    def __getitem__(self, idx):
        x = self.series[idx:idx + self.input_window]
        y = self.series[idx + self.input_window : idx + self.input_window + self.forecast_horizon]

        # shape for LSTM: (seq_len, input_size)
        x = x.unsqueeze(-1) #(input_window, 1)

        if self.forecast_horizon == 1:
            y = y.squeeze(0)
        return x, y
    

In [14]:
input_window = 30
forecast_horizon = 1
batch_size = 32

train_dataset = TimeSeriesDataset(train_series, input_window, forecast_horizon)
test_dataset = TimeSeriesDataset(test_series, input_window, forecast_horizon)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


In [15]:
class LSTMForecaster(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=2, output_size=1):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size = hidden_size,
            num_layers = num_layers,
            batch_first=True 
        )
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        #x: (batch, seq_len, input_size)
        out, _ = self.lstm(x)
        last_hidden = out[:,-1,:]
        pred = self.fc(last_hidden)
        return pred.squeeze(-1)
        

In [19]:

class GRUForecaster(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=2):
        super().__init__()

        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # x: (batch, seq_len, 1)

        out, _ = self.gru(x)

        last_hidden = out[:, -1, :]
        pred = self.fc(last_hidden)

        return pred.squeeze(-1)

raw time series input looks like:
```
(batch, seq_len, input_dim)
```

But Transformer expects:
```
(batch, seq_len, d_model)
```
- `d_model` is a hyperparameter: how much capacity the model has per timestep
- `n_head` is number of attention heads
- `num_layers` is the number of stacked Transformer encoder blocks, where each block has multi-head attention, feedforward network, and residual+layer norm
- `dim_feedforward` is the hidden size of per-timestep MLP 

```
d_model = 64
nhead = 4

head_dim = 64/4 = 16 
```

Each head learns different relationships:
- short-term pattern
- long-term pattern
- seasonality
- trend shift
- anomaly signal


In [18]:
class TransformerForecaster(nn.Module):
    def __init__(self,
                 input_dim=1,
                 d_model=64,
                 nhead=4,
                 num_layers=2,
                 dim_feedforward=128):

        super().__init__()

        self.input_proj = nn.Linear(input_dim, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            batch_first=True
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.fc = nn.Linear(d_model, 1)

    def forward(self, x):
        # x: (batch, seq_len, 1)
        x = self.input_proj(x)
        h = self.encoder(x)
        last = h[:, -1, :]
        pred = self.fc(last)

        return pred.squeeze(-1)

In [16]:
model = LSTMForecaster()
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

num_epochs = 20
for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0 
    for x_batch, y_batch in train_loader:
        #x_batch: (batch, input_window, 1)
        #y_batch: (batch,)
        optimizer.zero_grad()

        preds = model(x_batch)
        loss = loss_fn(preds, y_batch)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * x_batch.size(0)
    train_loss /= len(train_loader.dataset)

    model.eval()
    test_loss = 0.0
    with torch.no_grad():
        for x_batch, y_batch in test_loader:
            preds = model(x_batch)
            loss = loss_fn(preds, y_batch)
            test_loss += loss.item() * x_batch.size(0)
    test_loss /= len(test_loader.dataset)
    print(f"Epoch {epoch+1:02d} | train_loss={train_loss:.4f} | test_loss={test_loss:.4f}")

Epoch 01 | train_loss=0.3996 | test_loss=0.0763
Epoch 02 | train_loss=0.0500 | test_loss=0.0416
Epoch 03 | train_loss=0.0288 | test_loss=0.0256
Epoch 04 | train_loss=0.0226 | test_loss=0.0220
Epoch 05 | train_loss=0.0189 | test_loss=0.0186
Epoch 06 | train_loss=0.0167 | test_loss=0.0170
Epoch 07 | train_loss=0.0147 | test_loss=0.0168
Epoch 08 | train_loss=0.0145 | test_loss=0.0149
Epoch 09 | train_loss=0.0136 | test_loss=0.0154
Epoch 10 | train_loss=0.0140 | test_loss=0.0143
Epoch 11 | train_loss=0.0146 | test_loss=0.0196
Epoch 12 | train_loss=0.0162 | test_loss=0.0171
Epoch 13 | train_loss=0.0152 | test_loss=0.0139
Epoch 14 | train_loss=0.0142 | test_loss=0.0187
Epoch 15 | train_loss=0.0151 | test_loss=0.0143
Epoch 16 | train_loss=0.0138 | test_loss=0.0129
Epoch 17 | train_loss=0.0135 | test_loss=0.0141
Epoch 18 | train_loss=0.0141 | test_loss=0.0135
Epoch 19 | train_loss=0.0138 | test_loss=0.0169
Epoch 20 | train_loss=0.0146 | test_loss=0.0133
